In [ ]:
# Setup and IBM backend connection

from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from time import sleep
from broadcasting import generate_qiskit_circuit, add_fidelity
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datetime import datetime

np.set_printoptions(linewidth=200, precision=3, suppress=True)

service = QiskitRuntimeService(name="mprest1")

In [ ]:
# Protocol parameters and circuit construction
M_senders = 1
N_receivers = 2
thetas = [np.pi / 6]
use_qec = True
tau_values = np.array([0, 500, 1000, 2000])  # delay sweep in backend dt units
shots = 1024

circuit = generate_qiskit_circuit(M_senders, N_receivers, thetas, use_receiver_qec_513=use_qec)
circuit, reg_name, phi = add_fidelity(circuit, N=N_receivers, thetas=thetas)

tau_param = next(p for p in circuit.parameters if p.name == "tau")
bound_circuits = [circuit.assign_parameters({tau_param: int(tau)}) for tau in tau_values]

print(f"M={M_senders}, N={N_receivers}, thetas={thetas}")
print(f"QEC: {use_qec}, target phi={phi:.4f}")
print(f"Fidelity register: '{reg_name}' ({N_receivers} bits)")
print(f"Circuit depth: {circuit.depth()}, qubits: {circuit.num_qubits}")
circuit.draw("mpl")

In [ ]:
# Transpile and submit to backend
backend = service.least_busy(simulator=False, operational=True)
print(f"Backend: {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuits = [pm.run(c) for c in bound_circuits]

sampler = Sampler(mode=backend)
pubs = [(isc, None) for isc in isa_circuits]
job = sampler.run(pubs, shots=shots)
print(f"Job ID: {job.job_id()}")
sleep(10)
print(f"Status: {job.status()}")

In [ ]:
# Extract results, compute fidelities, save to file
results = job.result()
fidelity_sweep = {}

for tau, pub_result in zip(tau_values, results):
    counts = getattr(pub_result.data, reg_name).get_counts()
    total = sum(counts.values())

    # Fidelity register bits are big-endian within the register:
    # bit[0] = MSB = receiver N-1, bit[N-1] = LSB = receiver 0
    fidelities = []
    for i in range(N_receivers):
        p0 = sum(v for bitstr, v in counts.items() if bitstr[N_receivers - 1 - i] == "0") / total
        fidelities.append(p0)

    fidelity_sweep[int(tau)] = {"fidelities": fidelities, "counts": counts}

# Print results
print(f"{'tau':>6}  " + "  ".join(f"recv_{i}" for i in range(N_receivers)))
print("-" * (8 + 8 * N_receivers))
for tau, data in fidelity_sweep.items():
    fids = "  ".join(f"{f:.4f}" for f in data["fidelities"])
    print(f"{tau:>6}  {fids}")

# Save to timestamped JSON
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outfile = results_dir / f"run_{timestamp}.json"

run_data = {
    "timestamp": datetime.now().isoformat(),
    "job_id": job.job_id(),
    "backend": backend.name,
    "shots": shots,
    "protocol": {
        "M": M_senders,
        "N": N_receivers,
        "thetas": [float(t) for t in thetas],
        "use_qec": use_qec,
        "target_phi": float(phi),
    },
    "tau_values": [int(t) for t in tau_values],
    "results": {
        str(tau): {
            "fidelities": data["fidelities"],
            "counts": data["counts"],
        }
        for tau, data in fidelity_sweep.items()
    },
}

with open(outfile, "w") as f:
    json.dump(run_data, f, indent=2)
print(f"\nResults saved to {outfile}")